In [2]:
import os
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import celloracle as co
from tqdm.notebook import tqdm  # 使用 notebook 版本的进度条
import warnings
import scanpy as sc

warnings.filterwarnings("ignore")

# ====================================================
# 1. 设定当前路径与核心文件名
# ====================================================
WORK_DIR = "/media/zenglab/result/qingyun/New_test"
os.chdir(WORK_DIR)

H5AD_FILE = "processed_data_k3_adaptive.h5ad"
EDGES_FILE = "pruned_edges.csv"  

#  核心参数
CLUSTER_COL = "rna_nn_alg1_label3"      
EMBEDDING_NAME = "X_umap"    

# ====================================================
# 2. 加载单细胞数据与初始化 Oracle
# ====================================================
print(f"正在加载表达矩阵: {H5AD_FILE} ...")
adata = ad.read_h5ad(H5AD_FILE)

print("正在初始化 CellOracle 对象...")
oracle = co.Oracle()

print("正在基于批次矫正后的 X_pca_harmony 计算细胞邻接图...")
# 明确指定使用 harmony 的结果来找邻居，这样算出来的 UMAP 最准
sc.pp.neighbors(adata, use_rep='X_pca_harmony', n_neighbors=15)

print("正在计算 UMAP 降维坐标...")
sc.tl.umap(adata)

print("✅ UMAP 补齐完成！目前的降维坐标有:", adata.obsm.keys())

oracle.import_anndata_as_raw_count(adata=adata,
                                   cluster_column_name=CLUSTER_COL,
                                   embedding_name=EMBEDDING_NAME)

print(f"✅ 数据加载成功！包含 {adata.shape[0]} 个细胞，{adata.shape[1]} 个基因。")

1 package does not meet CellOracle requirement.
 Your jupyter version is not_found. Please install jupyter
正在加载表达矩阵: processed_data_k3_adaptive.h5ad ...
正在初始化 CellOracle 对象...
正在基于批次矫正后的 X_pca_harmony 计算细胞邻接图...
正在计算 UMAP 降维坐标...
✅ UMAP 补齐完成！目前的降维坐标有: KeysView(AxisArrays with keys: X_pca, X_pca_harmony, spatial, X_umap)
✅ 数据加载成功！包含 28915 个细胞，3019 个基因。


In [3]:
import numpy as np

# 1. 计算 K 值
n_cells = oracle.adata.shape[0]
k = int(n_cells * 0.025)
k = max(20, min(k, 100))

# 2. 手动关联 PCA 坐标
# 注意：这里必须保证 X_pca_harmony 的维度 >= 50
oracle.pcs = oracle.adata.obsm['X_pca_harmony'] 

print(f"📊 细胞数: {n_cells}, 确定的 k: {k}")
print("⏳ 正在基于 Harmony 空间执行 Balanced KNN 插补...")

# 3. 执行插补
oracle.knn_imputation(n_pca_dims=50, 
                      k=k, 
                      balanced=True, 
                      b_sight=k*8, 
                      b_maxl=k*4, 
                      n_jobs=8) 

print("✅ KNN 插补完成！")

📊 细胞数: 28915, 确定的 k: 100
⏳ 正在基于 Harmony 空间执行 Balanced KNN 插补...
✅ KNN 插补完成！


In [4]:
import pandas as pd
import celloracle as co

edges_df = pd.read_csv(EDGES_FILE)

df_links = edges_df.rename(columns={
    "RBP": "source",
    "TG": "target",
    "final_score": "coef_mean"
})

clusters = sorted(oracle.adata.obs[oracle.cluster_column_name].unique())

links = co.Links(name=oracle.cluster_column_name)
links.links_dict     = {c: df_links.copy() for c in clusters}
links.filtered_links = {c: df_links.copy() for c in clusters}
links.cluster        = clusters

oracle.get_cluster_specific_TFdict_from_Links(links_object=links)

# ✅ 把 cluster_specific_TFdict 合并进 TFdict
unified_TFdict = {}
for cluster, tfdict in oracle.cluster_specific_TFdict.items():
    for tg, regs in tfdict.items():
        if tg not in unified_TFdict:
            unified_TFdict[tg] = set()
        unified_TFdict[tg].update(regs)

unified_TFdict = {tg: list(regs) for tg, regs in unified_TFdict.items()}
oracle.addTFinfo_dictionary(unified_TFdict)
print(f"✅ TFdict 导入完成：{len(oracle.TFdict)} 个靶基因")

# 拟合 GRN
oracle.fit_GRN_for_simulation(alpha=10, use_cluster_specific_TFdict=True)

# 验证
print(oracle)

first_cluster = list(oracle.coef_matrix_per_cluster.keys())[0]
mat = oracle.coef_matrix_per_cluster[first_cluster]
print(f"\ncoef_matrix 类型: {type(mat)}")
print(f"coef_matrix 列名前5（应为RBP名）: {list(mat.columns[:5])}")
print(f"coef_matrix 行索引前5（应为TG名）: {list(mat.index[:5])}")

# 抽查一个 RBP 是否在 coef_matrix 列中
test_rbp = edges_df['RBP'].iloc[0]
print(f"\n抽查 RBP '{test_rbp}' 是否在 coef_matrix 列中: {test_rbp in mat.columns}")

✅ TFdict 导入完成：2254 个靶基因


  0%|          | 0/29 [00:00<?, ?it/s]

Oracle object

Meta data
    celloracle version used for instantiation: 0.22.0
    n_cells: 28915
    n_genes: 3019
    cluster_name: rna_nn_alg1_label3
    dimensional_reduction_name: X_umap
    n_target_genes_in_TFdict: 2254 genes
    n_regulatory_in_TFdict: 556 genes
    n_regulatory_in_both_TFdict_and_scRNA-seq: 556 genes
    n_target_genes_both_TFdict_and_scRNA-seq: 2254 genes
    k_for_knn_imputation: 100
Status
    Gene expression matrix: Ready
    BaseGRN: Ready
    PCA calculation: Done
    Knn imputation: Done
    GRN calculation for simulation: Done


coef_matrix 类型: <class 'pandas.core.frame.DataFrame'>
coef_matrix 列名前5（应为RBP名）: ['1700019D03Rik', 'A2m', 'AW551984', 'Aamp', 'Abat']
coef_matrix 行索引前5（应为TG名）: ['1700019D03Rik', 'A2m', 'AW551984', 'Aamp', 'Abat']

抽查 RBP 'Acaa1a' 是否在 coef_matrix 列中: True


In [ ]:
# ====================================================
# 6. 批量执行 RBP 虚拟敲除与细胞状态偏移预测
# ====================================================
# 获取网络中的所有 RBP
raw_rbp_list = edges_df['RBP'].unique().tolist()
# 严格过滤：只保留那些在单细胞表达矩阵中实际存在的基因
rbps_to_knockout = [rbp for rbp in raw_rbp_list if rbp in oracle.adata.var_names]

# 如果你想先拿 2 个基因测测速度，可以取消下面这行的注释
# rbps_to_knockout = rbps_to_knockout[:2] 

print(f"🎯 最终确认共有 {len(rbps_to_knockout)} 个有效 RBP 即将进行敲除模拟...")

# 创建保存图片的专门文件夹
OUT_DIR = os.path.join(WORK_DIR, "Oracle_KO_Results")
os.makedirs(OUT_DIR, exist_ok=True)

# 【参数调整】箭头的视觉缩放比例：数值越小，箭头越长。
# 如果画出来的图里只有小黑点，改小 (如 10)；如果箭头太长糊成一团，改大 (如 50)
SCALE_SIM = 25 

for rbp in tqdm(rbps_to_knockout, desc="Knocking out RBPs"):
    try:
        # 1. 模拟物理断流：将真实单细胞数据中该 RBP 的表达量清零，并向后传播信号
        oracle.simulate_shift(perturb_condition={rbp: 0.0}, n_propagation=3)
        
        # 2. 估计转移概率 (最耗时)：在多维空间中寻找敲除后的细胞最像现在的谁
        oracle.estimate_transition_prob(n_neighbors=200, knn_random=True, sampled_fraction=1)
        
        # 3. 计算在 UMAP 二维平面上的实际位移向量
        oracle.calculate_embedding_shift(sigma_corr=0.05)
        
        # 4. 绘图出图
        fig, ax = plt.subplots(1, 2, figsize=[16, 7])
        
        # 左图：基于你真实权重的敲除结果
        oracle.plot_simulation_flow_on_grid(scale=SCALE_SIM, ax=ax[0])
        ax[0].set_title(f"Simulation: {rbp} Knockout", fontsize=16, fontweight='bold')
        
        # 右图：完全随机的网络对照组 (用于证明左图的走向具有生物学特异性)
        oracle.plot_simulation_flow_random_on_grid(scale=SCALE_SIM, ax=ax[1])
        ax[1].set_title("Randomized GRNs Control", fontsize=16)
        
        plt.tight_layout()
        
        # 保存图片
        save_path = os.path.join(OUT_DIR, f"KO_VectorField_{rbp}.png")
        fig.savefig(save_path, dpi=200, bbox_inches='tight')
        
        # 彻底关闭图表，
        plt.close(fig)
        
    except Exception as e:
        print(f"\n 处理 RBP '{rbp}' 时跳过。原因: {e}")

print(f"\n🎉 批量敲除实验大功告成！所有极其震撼的细胞流向图均已保存至文件夹：{OUT_DIR}")

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# ====================================================
# 1. 文件夹与参数设置
# ====================================================
# 创建专门存放表达量对比图的文件夹
OUT_DIR_EXPR = os.path.join(WORK_DIR, "cellorcle_KO_Expression_Plots_V2")
os.makedirs(OUT_DIR_EXPR, exist_ok=True)
raw_rbp_list = edges_df['RBP'].unique().tolist()
rbps_to_knockout = [rbp for rbp in raw_rbp_list if rbp in oracle.adata.var_names]

# 自动挑选该 RBP 下游调控强度 Top N 的靶基因进行展示
top_n_targets = 8 

# 如果只想看某个特定亚群的改变，修改这里 (如 "GMP_0")；None 表示全景扫描
target_cluster = None 

print(f"🚀 开始批量敲除并生成靶基因表达量对比图，共 {len(rbps_to_knockout)} 个 RBP...")

# 获取目标细胞群的布尔索引
if target_cluster is not None:
    cell_idx = oracle.adata.obs[CLUSTER_COL] == target_cluster
else:
    cell_idx = np.ones(oracle.adata.shape[0], dtype=bool)

# 全局绘图风格
sns.set_theme(style="ticks", font_scale=1.1)

# ====================================================
# 2. 批量循环执行
# ====================================================
for rbp in tqdm(rbps_to_knockout, desc="Batch KO Expression"):
    try:
        # 提取该 RBP 调控的所有下游基因
        targets = edges_df[edges_df['RBP'] == rbp].copy()
        
        # 根据调控强度 (final_score) 的绝对值从大到小排序，取前 N 个
        targets['abs_score'] = targets['final_score'].abs()
        top_targets = targets.sort_values(by='abs_score', ascending=False).head(top_n_targets)['TG'].tolist()
        
        # 过滤掉不在单细胞矩阵中的基因（防止报错）
        valid_targets = [g for g in top_targets if g in oracle.adata.var_names]
        
        if not valid_targets:
            continue # 如果没有有效的下游靶基因，跳过该 RBP
            
        # --- B. 核心步骤：执行虚拟敲除 ---
        # 注意：这里我们只模拟基因表达变化，不需要计算费时的 UMAP 转移概率！所以速度极快！
        oracle.simulate_shift(perturb_condition={rbp: 0.0}, n_propagation=3)
        
        # --- C. 提取表达量数据 ---
        df_list = []
        for gene in valid_targets:
            # 提取敲除前 (WT)
            expr_wt = oracle.adata[cell_idx, gene].layers["imputed_count"].copy()
            if hasattr(expr_wt, "toarray"): expr_wt = expr_wt.toarray()
            expr_wt = expr_wt.flatten()
            
            # 提取敲除后 (KO)
            expr_ko = oracle.adata[cell_idx, gene].layers["simulated_count"].copy()
            if hasattr(expr_ko, "toarray"): expr_ko = expr_ko.toarray()
            expr_ko = expr_ko.flatten()
            
            temp_df = pd.DataFrame({
                "Expression": np.concatenate([expr_wt, expr_ko]),
                "Condition": ["WT"] * len(expr_wt) + [f"{rbp} KO"] * len(expr_ko),
                "Gene": gene
            })
            df_list.append(temp_df)
            
        plot_df = pd.concat(df_list, ignore_index=True)
        
        # --- D. 绘图与出图保存 ---
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        
        # 图 A: 柱状图
        sns.barplot(data=plot_df, x="Gene", y="Expression", hue="Condition", 
                    palette=["#4C72B0", "#C44E52"], capsize=.1, ax=axes[0])
        title_suffix = f" in {target_cluster}" if target_cluster else " in All Cells"
        axes[0].set_title(f"Mean Expression (WT vs {rbp} KO){title_suffix}", fontweight='bold')
        axes[0].set_ylabel("Expression")
        
        # 图 B: 阴阳小提琴图
        sns.violinplot(data=plot_df, x="Gene", y="Expression", hue="Condition", split=True, 
                       inner="quart", palette=["#4C72B0", "#C44E52"], ax=axes[1])
        axes[1].set_title(f"Expression Distribution{title_suffix}", fontweight='bold')
        axes[1].set_ylabel("")
        
        sns.despine()
        plt.tight_layout()
        
        # 保存图片到指定文件夹
        save_path = os.path.join(OUT_DIR_EXPR, f"Expression_KO_{rbp}.png")
        plt.savefig(save_path, dpi=200)
        
        # 必须关闭画板释放内存，否则跑几十张图就会内存溢出死机！
        plt.close(fig)
        
    except Exception as e:
        print(f"\n⚠️ 处理 RBP '{rbp}' 时跳过，原因: {e}")

print(f"\n🎉 完美！批量敲除与表达量对比图全自动生成完毕！快去文件夹 '{OUT_DIR_EXPR}' 里收图吧！")

🚀 开始批量敲除并生成靶基因表达量对比图，共 556 个 RBP...


Batch KO Expression:   0%|          | 0/556 [00:00<?, ?it/s]


🎉 完美！批量敲除与表达量对比图全自动生成完毕！快去文件夹 '/media/zenglab/result/qingyun/New_test/cellorcle_KO_Expression_Plots' 里收图吧！


In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

summary_records = []

print(f"🚀 开始批量敲除汇总，共 {len(rbps_to_knockout)} 个 RBP...")

if target_cluster is not None:
    cell_idx = oracle.adata.obs[CLUSTER_COL] == target_cluster
else:
    cell_idx = np.ones(oracle.adata.shape[0], dtype=bool)

for rbp in tqdm(rbps_to_knockout, desc="Batch KO Summary"):
    try:
        targets = edges_df[edges_df['RBP'] == rbp].copy()
        targets['abs_score'] = targets['final_score'].abs()
        top_targets = targets.sort_values(by='abs_score', ascending=False).head(top_n_targets)['TG'].tolist()
        valid_targets = [g for g in top_targets if g in oracle.adata.var_names]

        if not valid_targets:
            continue

        oracle.simulate_shift(perturb_condition={rbp: 0.0}, n_propagation=3)

        for gene in valid_targets:
            expr_wt = oracle.adata[cell_idx, gene].layers["imputed_count"].copy()
            if hasattr(expr_wt, "toarray"): expr_wt = expr_wt.toarray()
            expr_wt = expr_wt.flatten()

            expr_ko = oracle.adata[cell_idx, gene].layers["simulated_count"].copy()
            if hasattr(expr_ko, "toarray"): expr_ko = expr_ko.toarray()
            expr_ko = expr_ko.flatten()

            mean_wt = expr_wt.mean()
            mean_ko = expr_ko.mean()
            delta = mean_ko - mean_wt
            delta_pct = (delta / mean_wt * 100) if mean_wt != 0 else np.nan

            summary_records.append({
                "KO_RBP": rbp,
                "Target_Gene": gene,
                "Mean_WT": round(mean_wt, 4),
                "Mean_KO": round(mean_ko, 4),
                "Delta": round(delta, 4),
                "Delta_pct": round(delta_pct, 2) if not np.isnan(delta_pct) else np.nan,
                "Abs_Delta": round(abs(delta), 4)
            })

    except Exception as e:
        print(f"\n⚠️ 处理 RBP '{rbp}' 时跳过，原因: {e}")

summary_df = pd.DataFrame(summary_records)
summary_df = summary_df.sort_values(by=["KO_RBP", "Abs_Delta"], ascending=[True, False])

summary_path = os.path.join(WORK_DIR, "KO_summary_expression_change.csv")
summary_df.to_csv(summary_path, index=False)

print(f"\n✅ 汇总表格已保存至: {summary_path}")
print(f"📊 共记录 {len(summary_df)} 条记录，涉及 {summary_df['KO_RBP'].nunique()} 个 RBP")
print(summary_df.head(10))

In [1]:
import celloracle as co
import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

print(f"🎉 破茧成蝶！CellOracle 当前版本: {co.__version__}")
print(f"✅ Scanpy 当前版本: {sc.__version__}")

/home/qingyun/.local/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


🎉 破茧成蝶！CellOracle 当前版本: 0.22.0
✅ Scanpy 当前版本: 1.9.3


In [28]:
import pandas as pd

# 1. 确认 fit 真的完成了
print("=== oracle 状态 ===")
print(oracle)

# 2. 检查 coef_matrix_per_cluster 是否存在且有内容
print("\n=== coef_matrix_per_cluster ===")
if hasattr(oracle, 'coef_matrix_per_cluster'):
    for cluster, mat in oracle.coef_matrix_per_cluster.items():
        print(f"  {cluster}: type={type(mat)}, shape={mat.shape if hasattr(mat, 'shape') else 'N/A'}, columns前5={list(mat.columns[:5]) if hasattr(mat, 'columns') else 'N/A'}")
        break  # 只看第一个
else:
    print("  ❌ coef_matrix_per_cluster 不存在！")

# 3. 检查 active_regulatory_genes
print("\n=== active_regulatory_genes ===")
if hasattr(oracle, 'active_regulatory_genes'):
    regs = oracle.active_regulatory_genes
    print(f"  数量: {len(regs)}, 前5个: {regs[:5]}")
else:
    print("  ❌ active_regulatory_genes 不存在！")

=== oracle 状态 ===
Oracle object

Meta data
    celloracle version used for instantiation: 0.22.0
    n_cells: 28915
    n_genes: 3019
    cluster_name: rna_nn_alg1_label3
    dimensional_reduction_name: X_umap
    n_target_genes_in_TFdict: 0 genes
    n_regulatory_in_TFdict: 0 genes
    n_regulatory_in_both_TFdict_and_scRNA-seq: 0 genes
    n_target_genes_both_TFdict_and_scRNA-seq: 0 genes
    k_for_knn_imputation: 100
Status
    Gene expression matrix: Ready
    BaseGRN: Not imported
    PCA calculation: Done
    Knn imputation: Done
    GRN calculation for simulation: Done


=== coef_matrix_per_cluster ===
  AC1: type=<class 'pandas.core.frame.DataFrame'>, shape=(3019, 3019), columns前5=['1700019D03Rik', 'A2m', 'AW551984', 'Aamp', 'Abat']

=== active_regulatory_genes ===
  数量: 556, 前5个: ['Acaa1a', 'Acin1', 'Acly', 'Aco2', 'Actn1']


In [29]:
import inspect
import celloracle as co
print(inspect.getsource(co.Oracle.simulate_shift))

    def simulate_shift(self, perturb_condition=None, GRN_unit=None,
                       n_propagation=3, ignore_warning=False, use_randomized_GRN=False, clip_delta_X=False):
        """
        Simulate signal propagation with GRNs. Please see the CellOracle paper for details.
        This function simulates a gene expression pattern in the near future.
        Simulated values will be stored in anndata.layers: ["simulated_count"]


        The simulation use three types of data.
        (1) GRN inference results (coef_matrix).
        (2) Perturb_condition: You can set arbitrary perturbation condition.
        (3) Gene expression matrix: The simulation starts from imputed gene expression data.

        Args:
            perturb_condition (dictionary): condition for perturbation.
               if you want to simulate knockout for GeneX, please set [perturb_condition={"GeneX": 0.0}]
               Although you can set any non-negative values for the gene condition, avoid setting bio